<a href="https://colab.research.google.com/github/boluwatifeakintayo/boluwatife-FLYRANKAI-repo/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/boluwatifeakintayo/boluwatife-FLYRANKAI-repo/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
!git clone https://github.com/boluwatifeakintayo/boluwatife-FLYRANKAI-repo.git
%cd boluwatife-FLYRANKAI-repo

Cloning into 'boluwatife-FLYRANKAI-repo'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 129 (delta 44), reused 100 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 1.85 MiB | 10.21 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/boluwatife-FLYRANKAI-repo


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

#Classification
#####I'm predicting a yes/no answer for each individual page ("is this page declining?"), not ordering pages into a priority line (ranking) and not grouping pages with no target (clustering).


In [5]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"{df.shape[0]} rows, {df.shape[1]} columns")
df.head(5)

30000 rows, 44 columns


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Target or proxy

## Q2 — Target or Proxy
**Answer:** `is_declining = (trend_direction == "down")`

**Where the label comes from:** It's built on `trend_pct`, an observed, measured
percent change in a page's real traffic. But the category boundaries are a
*defined rule* on top of that measurement — checked directly: `down` = trend_pct
below -20%, `stable` = trend_pct between -20% and +20% (so a page can drop up to
20% and still be called "stable"), and `up` = anything above +20%. `flat` and `new`
have no trend_pct at all, meaning those pages likely lack enough history to
calculate a percent change.

So I'm treating only the `down` category (54% of all pages — 16,262 of 30,000) as
the positive class. This is a real, non-trivial slice of the data, not a rare
edge case.

In [6]:
# Count how many pages fall into each trend category
# This shows us: is "down" rare or common? What other categories exist?
print(df["trend_direction"].value_counts())

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [7]:
# For EACH trend category, show the min/max/average of the actual percent change (trend_pct)
# This lets us see the real cutoff someone used to decide what counts as "down" vs "stable" vs "up"
df.groupby("trend_direction")["trend_pct"].describe()

,count,mean,std,min,25%,50%,75%,max
trend_direction,,,,,,,,
down,16262.0,-58.113830,23.488605,-100.0,-75.9,-55.60,-38.5,-20.0
flat,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
new,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
stable,5962.0,-3.185944,11.054723,-20.0,-12.7,-3.80,5.0,20.0
up,4388.0,190.673997,1145.029477,20.0,36.5,62.55,123.1,44900.0


In [9]:
# Build the target: 1 if trend_direction is "down", 0 otherwise
# We verified above that "down" specifically means trend_pct < -20%, so this is a
# clean, non-ambiguous slice of the data — not overlapping with "stable"
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

# Confirm the split matches what value_counts showed us
print(df["is_declining"].value_counts())
print(f"Base rate: {df['is_declining'].mean():.1%} of pages are declining")

is_declining
1    16262
0    13738
Name: count, dtype: int64
Base rate: 54.2% of pages are declining


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## Q3 — Success Metric
**Answer:** ROC-AUC, plus precision and recall, compared against the 54.2% base rate.

**Why not just accuracy:** Since the base rate is 54.2% (verified in Q2), a model
that always guesses "declining" would already score ~54% accuracy while learning
nothing useful. Accuracy alone wouldn't reveal whether the model found real signal.

**ROC-AUC** measures how well the model separates declining from non-declining
pages across all possible decision thresholds — a value of 0.5 means "no better
than a coin flip," 1.0 means "perfect separation."

**Precision** answers: of the pages the model flags as declining, how many really
are? **Recall** answers: of all the pages that are truly declining, how many did
the model catch? Both matter here because missing a real decline (low recall)
means a page slips through unreviewed, while flagging too many false alarms (low
precision) wastes the content team's limited review time.

In [11]:
# The "always guess the majority class" baseline
# This simulates the laziest possible model: it just guesses "declining" every time
naive_accuracy = df["is_declining"].mean()  # same number as the base rate
print(f"A model that always guesses 'declining' would score {naive_accuracy:.1%} accuracy")
print("This is the bar any real model needs to beat — and why accuracy alone isn't enough proof of learning")

A model that always guesses 'declining' would score 54.2% accuracy
This is the bar any real model needs to beat — and why accuracy alone isn't enough proof of learning


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [12]:
# These are the columns relevant to OUR specific question: "is this page declining?"
# We're picking signals from Week 1 discovery work (position, CTR) plus a few
# obvious candidates (age, content type, word count) — and our target column
lane_columns = [
    "content_id",        # identifies which page this row is
    "position_tier",     # Week 1: strongly related to clicks
    "ctr",                # Week 1: click-through rate
    "content_type",       # what kind of page this is
    "word_count",         # Week 1: found NOT to matter much on its own
    "content_age_days",   # how old the content is
    "search_volume",      # Week 1: found NOT to predict visibility
    "trend_direction",    # the raw column our target comes from
    "is_declining"        # our built target (1 = declining, 0 = not)
]

# Build our lane's slice: same rows (pages), just narrowed to the columns we actually care about
lane_df = df[lane_columns]

print(f"{lane_df.shape[0]} rows, {lane_df.shape[1]} columns")
lane_df.head(10)

30000 rows, 9 columns


,content_id,position_tier,ctr,content_type,word_count,content_age_days,search_volume,trend_direction,is_declining
0,content_304f48230142,striking,0.76,keyword article,3221.0,187,10.0,down,1
1,content_a1fb4e703a9e,page_3_5,0.05,keyword article,2481.0,445,90.0,down,1
2,content_9aa793d4d895,page_3_5,0.09,keyword article,3515.0,141,0.0,down,1
3,content_331d6c4de07b,page_1,0.49,keyword article,NaN,463,10.0,stable,0
4,content_d99b7a2d90ca,page_3_5,0.13,keyword article,2803.0,263,0.0,down,1
5,content_d4084a4bc775,page_1,0.03,keyword article,3080.0,147,720.0,down,1
6,content_9a34b442b552,page_1,0.00,keyword article,3059.0,90,0.0,down,1
7,content_a63219c6e95a,page_3_5,0.06,keyword article,NaN,445,590.0,stable,0
8,content_5e6c160719bc,page_3_5,0.09,keyword article,3807.0,90,0.0,down,1
9,content_c27558df2b0c,page_1,0.16,keyword article,NaN,257,0.0,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statementThe signals that seem obvious for a rule turn out to be either
uninformative or too tangled together to separate with simple if/else logic.

- `search_volume` vs `impressions_90d` correlation ≈ 0.001 — a rule leaning on
  keyword popularity would be built on noise.
- Median `word_count` is nearly identical between declining (2909) and growing
  (2848) pages — a length-based cutoff wouldn't separate the groups at all.
- `position_tier` strongly predicts CTR (0.3548 at page_1 vs 0.0554 "deep"), but
  position alone can't distinguish a page that's *always* been low-ranked from
  one that's *actively declining* — the same position value means different
  things depending on the page's history.

None of these signals is individually enough to write a reliable if-statement,
and the ones that might combine well (position + CTR + age + content type,
weighted differently for different situations) aren't obvious to a human by
inspection. A model can learn these weighted combinations directly from the
data, which is exactly why the reference pipeline's learned model reached
Precision@50 ≈ 0.74 versus ≈ 0.24 for the hand-written rule.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.